In [ ]:
import json
import math
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.interpolate import CubicSpline
from tqdm import tqdm

PROJECT_ROOT = Path("/Users/jhonmichaelferrer/Documents/VSCode/Coreset/CoreSet-AI-Fitness-Tracker-ML-Pipeline")
INPUT_BASE = PROJECT_ROOT / "data" / "final"
OUTPUT_BASE = PROJECT_ROOT / "data" / "interpolated_json"

CATEGORIES = ["bench_press", "pull_up", "push_up", "squat"]
VISIBILITY_THRESHOLD = 0.5
REPAIR_FINAL_IN_PLACE = True

In [ ]:
def landmark_to_vector(landmark):
    if landmark is None:
        return [np.nan, np.nan, np.nan, np.nan]

    if isinstance(landmark, dict):
        values = [
            landmark.get("x", np.nan),
            landmark.get("y", np.nan),
            landmark.get("z", np.nan),
            landmark.get("visibility", np.nan),
        ]
    else:
        values = list(landmark[:4])
        values.extend([np.nan] * (4 - len(values)))

    xyz = np.asarray(values[:3], dtype=float)

    if np.isfinite(xyz).all() and np.allclose(xyz, 0.0):
        values[:3] = [np.nan, np.nan, np.nan]

    return values


def fill_series(values, fallback_values=None):
    values = np.asarray(values, dtype=float)
    time_indices = np.arange(len(values))
    valid_mask = np.isfinite(values)

    if valid_mask.sum() == 0 and fallback_values is not None:
        values = np.asarray(fallback_values, dtype=float)
        valid_mask = np.isfinite(values)

    if valid_mask.sum() > 3:
        valid_t = time_indices[valid_mask]
        valid_y = values[valid_mask]

        cs = CubicSpline(
            valid_t,
            valid_y,
            bc_type="natural",
            extrapolate=False,
        )

        filled = cs(time_indices)

        outside_mask = ~np.isfinite(filled)
        if outside_mask.any():
            filled[outside_mask] = np.interp(
                time_indices[outside_mask],
                valid_t,
                valid_y,
            )

        return filled

    if valid_mask.sum() > 1:
        return np.interp(
            time_indices,
            time_indices[valid_mask],
            values[valid_mask],
        )

    if valid_mask.sum() == 1:
        return np.full(len(values), values[valid_mask][0])

    return values


def fill_remaining_missing_values(landmark_array):
    coords = landmark_array[:, :, :3]

    global_fallback = np.nanmedian(
        np.where(np.isfinite(coords), coords, np.nan).reshape(-1, 3),
        axis=0,
    )

    global_fallback = np.where(
        np.isfinite(global_fallback),
        global_fallback,
        0.0,
    )

    for frame_idx in range(landmark_array.shape[0]):
        frame_coords = landmark_array[frame_idx, :, :3]

        frame_fallback = np.nanmedian(
            np.where(np.isfinite(frame_coords), frame_coords, np.nan),
            axis=0,
        )

        frame_fallback = np.where(
            np.isfinite(frame_fallback),
            frame_fallback,
            global_fallback,
        )

        missing_mask = ~np.isfinite(frame_coords)

        if missing_mask.any():
            landmark_array[frame_idx, :, :3] = np.where(
                missing_mask,
                frame_fallback,
                frame_coords,
            )

    visibility = landmark_array[:, :, 3]
    visibility = np.where(
        np.isfinite(visibility),
        visibility,
        VISIBILITY_THRESHOLD,
    )

    landmark_array[:, :, 3] = np.clip(visibility, 0.0, 1.0)

    return landmark_array

In [ ]:
def interpolate_skeletal_data(json_data, threshold=0.5, expected_landmarks=33):
    frames = json_data.get("frames", []) if isinstance(json_data, dict) else json_data

    landmark_array = np.full(
        (len(frames), expected_landmarks, 4),
        np.nan,
        dtype=float,
    )

    for frame_idx, frame in enumerate(frames):
        landmarks = frame.get("landmarks") if isinstance(frame, dict) else None

        if not landmarks:
            continue

        for lm_idx in range(min(expected_landmarks, len(landmarks))):
            landmark_array[frame_idx, lm_idx] = landmark_to_vector(landmarks[lm_idx])

    for lm_idx in range(expected_landmarks):
        visibility = landmark_array[:, lm_idx, 3]
        visible_mask = np.isfinite(visibility) & (visibility >= threshold)

        for axis in range(3):
            original_values = landmark_array[:, lm_idx, axis].copy()
            values = original_values.copy()
            values[~visible_mask] = np.nan

            landmark_array[:, lm_idx, axis] = fill_series(
                values,
                fallback_values=original_values,
            )

        landmark_array[:, lm_idx, 3] = fill_series(visibility)

    landmark_array = fill_remaining_missing_values(landmark_array)

    clean_frames = []

    for frame_idx, frame in enumerate(frames):
        clean_frame = dict(frame) if isinstance(frame, dict) else {"frame_index": frame_idx}
        clean_frame["landmarks"] = landmark_array[frame_idx].tolist()
        clean_frames.append(clean_frame)

    if isinstance(json_data, dict):
        clean_data = dict(json_data)
        clean_data["frames"] = clean_frames
        return clean_data

    return clean_frames

In [ ]:
def has_usable_coordinates(json_data):
    frames = json_data.get("frames", []) if isinstance(json_data, dict) else json_data

    for frame in frames:
        landmarks = frame.get("landmarks") if isinstance(frame, dict) else None

        if not landmarks:
            continue

        for landmark in landmarks:
            vector = landmark_to_vector(landmark)

            if np.isfinite(np.asarray(vector[:3], dtype=float)).any():
                return True

    return False


def data_to_landmark_array(json_data, expected_landmarks=33):
    frames = json_data.get("frames", []) if isinstance(json_data, dict) else json_data

    landmark_array = np.full(
        (len(frames), expected_landmarks, 4),
        np.nan,
        dtype=float,
    )

    for frame_idx, frame in enumerate(frames):
        landmarks = frame.get("landmarks") if isinstance(frame, dict) else None

        if not landmarks:
            continue

        for lm_idx in range(min(expected_landmarks, len(landmarks))):
            landmark_array[frame_idx, lm_idx] = landmark_to_vector(landmarks[lm_idx])

    return fill_remaining_missing_values(landmark_array)


def resample_landmark_array(source_array, target_frames):
    source_frames = source_array.shape[0]

    if source_frames == target_frames:
        return source_array.copy()

    source_t = np.linspace(0.0, 1.0, source_frames)
    target_t = np.linspace(0.0, 1.0, target_frames)

    result = np.zeros(
        (target_frames, source_array.shape[1], source_array.shape[2]),
        dtype=float,
    )

    for lm_idx in range(source_array.shape[1]):
        for axis in range(source_array.shape[2]):
            values = source_array[:, lm_idx, axis]
            result[:, lm_idx, axis] = np.interp(target_t, source_t, values)

    result[:, :, 3] = np.clip(result[:, :, 3], 0.0, 1.0)

    return result


def apply_fallback_trajectory(json_data, fallback_array):
    frames = json_data.get("frames", []) if isinstance(json_data, dict) else json_data
    landmark_array = resample_landmark_array(fallback_array, len(frames))

    clean_frames = []

    for frame_idx, frame in enumerate(frames):
        clean_frame = dict(frame) if isinstance(frame, dict) else {"frame_index": frame_idx}
        clean_frame["landmarks"] = landmark_array[frame_idx].tolist()
        clean_frames.append(clean_frame)

    if isinstance(json_data, dict):
        clean_data = dict(json_data)
        clean_data["frames"] = clean_frames
        return clean_data

    return clean_frames


def build_category_fallback(input_folder):
    for candidate_path in sorted(input_folder.glob("*.json")):
        with candidate_path.open("r") as f:
            candidate_data = json.load(f)

        if has_usable_coordinates(candidate_data):
            return data_to_landmark_array(candidate_data), candidate_path.name

    return None, None

In [ ]:
def landmark_to_output_list(landmark):
    if isinstance(landmark, dict):
        return [
            landmark.get("x", 0.0),
            landmark.get("y", 0.0),
            landmark.get("z", 0.0),
            landmark.get("visibility", 0.0),
        ]

    return list(landmark[:4])


def format_interpolated_output(clean_data):
    frames = clean_data.get("frames", clean_data) if isinstance(clean_data, dict) else clean_data
    output_frames = []

    for frame_idx, frame in enumerate(frames):
        output_frame = {
            "frame_index": frame.get("frame_index", frame_idx)
            if isinstance(frame, dict)
            else frame_idx,
            "landmarks": [
                landmark_to_output_list(landmark)
                for landmark in frame.get("landmarks", [])
            ],
        }

        output_frames.append(output_frame)

    if isinstance(clean_data, dict):
        output_data = {
            key: value
            for key, value in clean_data.items()
            if key != "frames"
        }

        output_data["frames"] = output_frames
        return output_data

    return output_frames

In [ ]:
stats = {
    cat: {
        "input": 0,
        "output": 0,
        "final_repaired": 0,
    }
    for cat in CATEGORIES
}

errors = []

for category in CATEGORIES:
    input_folder = INPUT_BASE / category
    output_folder = OUTPUT_BASE / category
    output_folder.mkdir(parents=True, exist_ok=True)

    files = sorted(input_folder.glob("*.json"))
    stats[category]["input"] = len(files)

    category_fallback, fallback_name = build_category_fallback(input_folder)

    if category_fallback is None:
        errors.append((category, "No usable category fallback file found"))
        continue

    for input_path in tqdm(files, desc=f"Processing {category}", leave=False):
        try:
            with input_path.open("r") as f:
                raw_data = json.load(f)

            frames = raw_data.get("frames", []) if isinstance(raw_data, dict) else raw_data
            original_len = len(frames)

            if original_len == 0:
                raise ValueError("No frames found")

            if has_usable_coordinates(raw_data):
                clean_data = interpolate_skeletal_data(
                    raw_data,
                    VISIBILITY_THRESHOLD,
                )
            else:
                clean_data = apply_fallback_trajectory(
                    raw_data,
                    category_fallback,
                )

                print(
                    f"Applied {category} fallback trajectory "
                    f"from {fallback_name} to {input_path.name}"
                )

            clean_frames = (
                clean_data.get("frames", [])
                if isinstance(clean_data, dict)
                else clean_data
            )

            if len(clean_frames) != original_len:
                raise ValueError(
                    f"Frame count changed from {original_len} to {len(clean_frames)}"
                )

            output_data = format_interpolated_output(clean_data)

            with (output_folder / input_path.name).open("w") as out_f:
                json.dump(output_data, out_f)

            stats[category]["output"] += 1

            if REPAIR_FINAL_IN_PLACE:
                with input_path.open("w") as final_f:
                    json.dump(clean_data, final_f)

                stats[category]["final_repaired"] += 1

        except Exception as e:
            errors.append((str(input_path), str(e)))

if errors:
    print(f"\nCompleted with {len(errors)} skipped files/folders. First 10:")

    for item, message in errors[:10]:
        print(f"- {item}: {message}")

In [ ]:
def audit_pose_json(path):
    with path.open("r") as f:
        data = json.load(f)

    frames = data.get("frames", data) if isinstance(data, dict) else data

    null_landmarks = 0
    zero_xyz_triplets = 0
    non_finite_values = 0
    dict_landmarks = 0
    video_path_keys = 0

    if isinstance(data, dict) and "video_path" in data:
        video_path_keys += 1

    for frame in frames:
        landmarks = frame.get("landmarks") if isinstance(frame, dict) else None

        if landmarks is None:
            null_landmarks += 1
            continue

        for landmark in landmarks:
            if landmark is None:
                null_landmarks += 1
                continue

            if isinstance(landmark, dict):
                dict_landmarks += 1
                values = [
                    landmark.get("x"),
                    landmark.get("y"),
                    landmark.get("z"),
                    landmark.get("visibility"),
                ]
            else:
                values = list(landmark[:4])

            values = [float(v) for v in values]

            non_finite_values += sum(
                not math.isfinite(v)
                for v in values
            )

            if all(math.isfinite(v) and v == 0.0 for v in values[:3]):
                zero_xyz_triplets += 1

    return {
        "null_landmarks": null_landmarks,
        "zero_xyz_triplets": zero_xyz_triplets,
        "non_finite_values": non_finite_values,
        "dict_landmarks": dict_landmarks,
        "video_path_keys": video_path_keys,
    }

In [ ]:
print("\n" + "=" * 78)
print(f"{'CORESET CATEGORY AUDIT REPORT':^78}")
print("=" * 78)
print(
    f"{'Category':<20} | "
    f"{'Input Files':<11} | "
    f"{'Output':<7} | "
    f"{'Final Repaired':<14}"
)
print("-" * 78)

total_in = 0
total_out = 0
total_repaired = 0

for cat, counts in stats.items():
    print(
        f"{cat.replace('_', ' ').title():<20} | "
        f"{counts['input']:<11} | "
        f"{counts['output']:<7} | "
        f"{counts['final_repaired']:<14}"
    )

    total_in += counts["input"]
    total_out += counts["output"]
    total_repaired += counts["final_repaired"]

print("-" * 78)
print(f"{'TOTAL':<20} | {total_in:<11} | {total_out:<7} | {total_repaired:<14}")
print("=" * 78)


for folder_name, folder_path in [
    ("final", INPUT_BASE),
    ("interpolated_json", OUTPUT_BASE),
]:
    totals = Counter()

    for json_path in folder_path.rglob("*.json"):
        totals.update(audit_pose_json(json_path))

    print(
        f"{folder_name}: "
        f"null landmarks={totals['null_landmarks']}, "
        f"all-zero XYZ triplets={totals['zero_xyz_triplets']}, "
        f"non-finite values={totals['non_finite_values']}, "
        f"dict landmarks={totals['dict_landmarks']}, "
        f"video_path keys={totals['video_path_keys']}"
    )